In [29]:
import pandas as pd
import numpy as np
import geopandas as gpd
import glob
import warnings
warnings.filterwarnings('ignore')

data_lists = glob.glob(r"D:\\02_사업관리\\자체분석\\01_생활인구\\*_tme_p2_*.csv", recursive=True)
g100m = gpd.read_file("추출영역.gpkg", layer="g100m_area")
gcode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\부산구군코드.csv", dtype={'gcode': str}, encoding="utf-8")
output_path = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260721_자연재난과\\집계결과\\"

def extract(data_lists, grid):
    res_lists = []
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str}, encoding="utf-8")
        # df = df[(df['hcode'].str.startswith('26260')) | (df['hcode'].str.startswith('26410')) | (df['hcode'].str.startswith('26470'))]
        df['std_ym'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:7]
        df['weekname'] = df['std_ymd'].astype('datetime64[ns]').dt.day_name('ko_KR')
        df['gcode'] = df['hcode'].str[:5]
        df = pd.merge(df, gcode, on="gcode", how="left")
        df = pd.merge(grid, df, left_on="gid", right_on="id", how="inner")
        res_lists.append(df)
    concated = pd.concat(res_lists, ignore_index=True)
    return concated

def aggregate(data_lists, grid):
    df = extract(data_lists, grid)
    df.drop('geometry', axis=1, inplace=True)
    time_dicts = dict((f'{h:02d}H', [f'h_t_{h:02d}', f'w_t_{h:02d}', f'v_t_{h:02d}']) for h in range(0, 24))
    for key, value in time_dicts.items():
        df[key] = df.loc[:, value].sum(axis=1) / 3
    
    keys = [key for key in time_dicts.keys()]
    group = df.groupby(['std_ym','weekname','gname','gid'], as_index=False)[keys].sum().round()
    group = pd.melt(group, id_vars=['std_ym','weekname','gname','gid'], value_vars=keys, var_name='times', value_name='pop')
    group = group[['std_ym','weekname','gname','gid','times','pop']]
    group.columns = ['기준연월','요일','구군명','gid','시간대','인구수']
    return group

aggregate(data_lists, g100m).to_csv(output_path + "2026-019_온천천 인근 시간대별 생활인구 현황.csv", sep="|",encoding="utf-8", index=False)

In [30]:
aggregate(data_lists, g100m).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1784160 entries, 0 to 1784159
Data columns (total 6 columns):
 #   Column  Dtype  
---  ------  -----  
 0   기준연월    object 
 1   요일      object 
 2   구군명     object 
 3   gid     object 
 4   시간대     object 
 5   인구수     float64
dtypes: float64(1), object(5)
memory usage: 81.7+ MB


In [ ]:
g100m = gpd.read_file("추출영역.gpkg", layer="g100m_area")
g100m.info()